In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, count, when, isnan, isnull, trim, length, current_timestamp, from_utc_timestamp, row_number, udf, lit
from pyspark.sql.types import StringType, BooleanType
from pyspark.sql.window import Window


def calcular_completude(df: DataFrame, colunas_obrigatorias: list) -> dict:
    total_registros = df.count()
    metricas = {}
    for coluna in colunas_obrigatorias:
        nao_nulos = df.filter(col(coluna).isNotNull()).count()
        taxa_completude = (nao_nulos / total_registros * 100) if total_registros > 0 else 0
        metricas[coluna] = {
            'total': total_registros,
            'preenchidos': nao_nulos,
            'nulos': total_registros - nao_nulos,
            'taxa_completude_%': round(taxa_completude, 2)
        }
    return metricas

def validar_precisao_numerica(df: DataFrame, coluna: str, min_val: float = None, max_val: float = None) -> DataFrame:
    condicao = col(coluna).isNotNull()
    if min_val is not None:
        condicao = condicao & (col(coluna) >= min_val)
    if max_val is not None:
        condicao = condicao & (col(coluna) <= max_val)
    return df.filter(condicao)

def remover_duplicados(df: DataFrame, chave_primaria: list, criterio_desempate: str = "hora_ingestao") -> DataFrame:
    window_spec = Window.partitionBy(chave_primaria).orderBy(col(criterio_desempate).desc())
    df_deduplicated = df.withColumn("row_num", row_number().over(window_spec)) \
                        .filter(col("row_num") == 1) \
                        .drop("row_num")
    return df_deduplicated

def adicionar_metadados_silver(df: DataFrame) -> DataFrame:
    return df.withColumn("data_processamento_silver", 
                         from_utc_timestamp(current_timestamp(), "America/Sao_Paulo"))



In [0]:

# COLOCAR LIMITE DA BASE SE QUISER AMOSTRA ==============================================================================================================


# Configuração
CATALOG = "workspace"
SCHEMA_BRONZE = "yelp_bronze"
SCHEMA_SILVER = "yelp_silver" 

# ========== TABELA 2: REVIEW ==========

table_name = "yelp_academic_dataset_review"
print(f"Processando tabela: {table_name}")
print("="*60)

# Leitura da tabela bronze 
df_review = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{table_name}")
print(f"Registros originais da review: {df_review.count()}")

# Join com tabela silver de business para pegar apenas as reviews referentes às categorias de interesse
df_business_silver = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.business")
df_review_food = df_review.join(
    df_business_silver.select("business_id", "food_category"),
    on="business_id",
    how="inner"
)
print(f"Registros após inner join com silver business: {df_review_food.count()}")

# Remove coluna text que não será usada
df_review_food = df_review_food.drop('text')
print("✓ Coluna removida: text")

# COMPLETUDE: Campos obrigatórios
colunas_obrigatorias = ['review_id', 'user_id', 'business_id', 'stars', 'date']
metricas_completude = calcular_completude(df_review_food, colunas_obrigatorias)

print("\n--- Métricas de COMPLETUDE ---")
for col_name, metricas in metricas_completude.items():
    print(f"  {col_name}: {metricas['taxa_completude_%']}% completo ({metricas['nulos']} nulos)")

# Filtra registros com campos obrigatórios preenchidos
df_review_food_clean = df_review_food.filter(
    col('review_id').isNotNull() & 
    col('user_id').isNotNull() & 
    col('business_id').isNotNull() &
    col('stars').isNotNull() &
    col('date').isNotNull()
)

print(f"\nApós filtro de completude: {df_review_food_clean.count()} registros")

# PRECISÃO: Validações
print("\n--- Validação de PRECISÃO ---")

# Stars: 1 a 5 (reviews não permitem 0 stars)
df_review_food_clean = validar_precisao_numerica(df_review_food_clean, 'stars', min_val=1, max_val=5)
print(f"  Stars (1-5): {df_review_food_clean.count()} registros válidos")

# Useful, funny, cool >= 0
for coluna in ['useful', 'funny', 'cool']:
    df_review_food_clean = df_review_food_clean.filter(
        col(coluna).isNull() | (col(coluna) >= 0)
    )
print(f"  Contadores (useful/funny/cool >=0): {df_review_food_clean.count()} registros válidos")

# Remoção de duplicados
print("\n--- Remoção de Duplicados ---")
df_review_food_clean = remover_duplicados(df_review_food_clean, ['review_id'])
print(f"Após deduplicação: {df_review_food_clean.count()} registros")

# Adiciona metadados silver
df_review_silver = adicionar_metadados_silver(df_review_food_clean)

# Salva na camada Silver
table_silver = f"{CATALOG}.{SCHEMA_SILVER}.review"
df_review_silver.write.mode("overwrite").saveAsTable(table_silver)

print(f"\n✓ Tabela silver criada: {table_silver}")
print(f"Total de registros silver: {df_review_silver.count()}")
print("="*60)

display(spark.table(table_silver).limit(20))

In [0]:
# Visualiza amostra dos dados limpos
print("AMOSTRA - Tabela Silver Review:")
print("="*60)

df_sample = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.review")

print(f"Total de reviews: {df_sample.count()}")

# Amostra de dados
print("\nPrimeiros 5 registros:")
display(df_sample.select(
    'review_id', 
    'user_id', 
    'business_id', 
    'stars',
    'date',
    'data_processamento_silver',
    'cool',
    'funny',
    'useful',
    'hora_ingestao',
    'food_category'
).limit(5))

# Distribuição de stars
print("\nDistribuição de Stars:")
display(df_sample.groupBy('stars').count().orderBy('stars'))